# This notebook is related to the Part of Speech task.

In [2]:
"""
This module processes ASR (Automatic Speech Recognition) transcriptions and 
compares them against reference transcriptions and corrected ASR transcriptions. 
It identifies types of edits (improvements, introduced errors, etc.) and analyzes 
the part-of-speech (POS) tags associated with these edits.
"""

'\nThis module processes ASR (Automatic Speech Recognition) transcriptions and \ncompares them against reference transcriptions and corrected ASR transcriptions. \nIt identifies types of edits (improvements, introduced errors, etc.) and analyzes \nthe part-of-speech (POS) tags associated with these edits.\n'

In [1]:
import sys
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from jiwer import RemovePunctuation
from dotenv import load_dotenv
import spacy
from tqdm import tqdm


!{sys.executable} -m pip install chat_gpt_asr
from chat_gpt_asr.alignment import *

# read env variables
load_dotenv()
Root = os.getenv("ROOT_PATH")

# load spacy
nlp = spacy.load("en_core_web_sm")

## Read dataset

In [5]:
# read dataset
path = os.path.join(Root, "results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json")

#path = "/home/mnaderi/Documents/thesis/chat-gpt-asr/results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json"

with open(path, "r") as f:
    data = json.load(f)
    
    transcriptions = [RemovePunctuation()(d["asr_transcription"]["text"]).lower().strip() for d in data]
    reference_transcriptions = [RemovePunctuation()(d["reference_transcription"]).lower().strip() for d in data]
    corrected_transcriptions = [RemovePunctuation()(d["corrected_asr_transcription"]).lower().strip() for d in data]

## Parts of speech

In [6]:

def identify_operation(a, l):
    """
    Identify the type of operation needed to transform the ASR transcription to the corrected transcription.
    
    Parameters:
    a (str): ASR transcription token.
    l (str): Corrected ASR transcription token.
    
    Returns:
    str: The operation type ('D' for deletion, 'I' for insertion, 'S' for substitution, '-' for no change).
    """
    if a != "" and l == "":
        return "D"
    elif a == "" and l != "":
        return "I"
    elif a != "" and l != "" and a != l:
        return "S"
    else:
        return '-'

def identify_edit_type_1(rr, aa, ll):
    """
    Identify the types of edits between reference, ASR, and corrected ASR transcriptions.
    
    Parameters:
    rr (list): List of tokens from the reference transcription.
    aa (list): List of tokens from the ASR transcription.
    ll (list): List of tokens from the corrected ASR transcription.
    
    Returns:
    tuple: A tuple containing edit types, edits, and operations.
    """
    edit_types = []
    operations = []
    edits = {"Improved":[],"IntroducedError":[],"LeftCorrect":[],"LeftIncorrect":[]}
    # rr, aa, ll = align3(ref, asr, llm, tokenizer_fn)
    if len(rr) == len(aa) == len(ll):
        for r,a,l in zip(rr,aa,ll):
            operation = identify_operation(a, l)
            operations.append(operation)
            if a == l == r:
                edit_types.append("LeftCorrect")  # left it correct
                edits["LeftCorrect"].append((a,l,r))
            elif a != l and l == r:
                edit_types.append("Improved")  # improve it
                edits["Improved"].append((a,l,r))
            elif a != r and l != r:
                edit_types.append("LeftIncorrect")  # left it incorrect
                edits["LeftIncorrect"].append((a,l,r))
            elif a == r and l != r:
                edit_types.append("IntroducedError")  # introducing an error
                edits["IntroducedError"].append((a,l,r))
    else:
        raise Exception
    return edit_types, edits, operations

def preprocess(ref, asr, llm, tokenizer_fn):
    """
    Preprocess the transcriptions to align and identify edits and part-of-speech tags.
    
    Parameters:
    ref (str): Reference transcription.
    asr (str): ASR transcription.
    llm (str): Corrected ASR transcription.
    tokenizer_fn (function): Function to tokenize the input strings.
    
    Returns:
    dict: A dictionary with edit types and their associated part-of-speech tags.
    """

    # list of tokens of aligned ref, asr, and llm 
    ref_aligned, asr_aligned, llm_aligned = align3(ref,asr,llm, tokenizer_fn=tokenizer_fn)

    # identify the edit_types and operations
    edit_types, edits, operations = identify_edit_type_1(ref_aligned, asr_aligned, llm_aligned)
    llm_aligned_str = " ".join([r if r else "*" for r in llm_aligned])
    asr_aligned_str = " ".join([r if r else "*" for r in asr_aligned])
    ref_aligned_str = " ".join([r if r else "*" for r in ref_aligned])
    
    # string of aligned ref, asr, and llm with '' replaced by *
    llm_doc = nlp(llm_aligned_str)
    ref_doc = nlp(ref_aligned_str)
    asr_doc = nlp(asr_aligned_str)
    assert len(llm_doc) == len(asr_doc) == len(ref_doc)

    d = {}
    for i in range(len(asr_doc)):

        operation = operations[i]
        edit_type = edit_types[i]
        a_token = asr_doc[i]
        a_pos = "MASKED" if a_token.text == "*" else a_token.pos_
        
        l_token = llm_doc[i]
        l_pos = "MASKED" if l_token.text == "*" else l_token.pos_

        # identify pos
        if operation == 'I':
            pos = l_pos
        elif operation == 'D':
            pos = a_pos
        elif operation == '-':
            pos = a_pos
        elif operation == 'S':
            pos = a_pos
    
        d[(edit_type, pos)] = d.get((edit_type, pos), 0) + 1
        # print(f"{operation=}, {edit_type=}, {a_pos=} {l_pos=}, {pos=}")

    return d

def tokenizer_fn(s):
    """
    Tokenize the input string using the Spacy model.
    
    Parameters:
    s (str): Input string to be tokenized.
    
    Returns:
    list: List of tokens from the input string.
    """
    return [token.text for token in nlp(s)]

In [10]:
import multiprocessing as mp

# parallelized approach
def process_triplet(input):
    """
    Process a triplet of transcriptions in parallel.
    
    Parameters:
    input (tuple): Tuple containing ASR transcription, reference transcription, and corrected ASR transcription.
    
    Returns:
    dict: A dictionary with edit types and their associated part-of-speech tags.
    """
    asr, ref, llm = input
    # process_id = os.getpid()
    # print(f"Process ID: {process_id}, Index: {index}")
    return preprocess(ref, asr, llm, tokenizer_fn)

num_processes = mp.cpu_count()  
with mp.Pool(processes=num_processes) as pool:
    # Map function to distribute tasks across processes
    results = list(
        pool.map(
            process_triplet, 
            zip(transcriptions, reference_transcriptions, corrected_transcriptions)
        )
    )


results

[{('LeftCorrect', 'VERB'): 4,
  ('LeftCorrect', 'ADP'): 4,
  ('LeftCorrect', 'PRON'): 4,
  ('LeftCorrect', 'DET'): 2,
  ('LeftCorrect', 'NOUN'): 3,
  ('LeftCorrect', 'ADJ'): 3,
  ('LeftCorrect', 'CCONJ'): 1,
  ('Improved', 'NOUN'): 1},
 {('LeftCorrect', 'PRON'): 1,
  ('LeftCorrect', 'VERB'): 1,
  ('LeftCorrect', 'ADP'): 2,
  ('LeftCorrect', 'DET'): 2,
  ('LeftCorrect', 'NOUN'): 3,
  ('Improved', 'ADV'): 1},
 {('LeftCorrect', 'ADP'): 2,
  ('LeftCorrect', 'NOUN'): 2,
  ('LeftCorrect', 'PRON'): 3,
  ('LeftCorrect', 'AUX'): 1,
  ('LeftCorrect', 'VERB'): 2,
  ('LeftCorrect', 'ADV'): 2,
  ('LeftCorrect', 'ADJ'): 1},
 {('LeftCorrect', 'PRON'): 9,
  ('LeftCorrect', 'AUX'): 3,
  ('LeftCorrect', 'VERB'): 5,
  ('LeftCorrect', 'DET'): 1,
  ('LeftCorrect', 'NOUN'): 4,
  ('LeftCorrect', 'CCONJ'): 1,
  ('Improved', 'PRON'): 1,
  ('LeftCorrect', 'ADP'): 1,
  ('LeftCorrect', 'ADV'): 1,
  ('Improved', 'VERB'): 1},
 {('LeftCorrect', 'INTJ'): 1,
  ('LeftIncorrect', 'MASKED'): 1,
  ('LeftCorrect', 'VERB'):

In [ ]:

# non-parallelized approach
# tokenizer_fn = lambda s: [token.text for token in nlp(s)]
# results = []
# for asr, ref, llm in tqdm(zip(transcriptions,reference_transcriptions,corrected_transcriptions), 
#                          total=len(transcriptions)):
#     d = preprocess(ref, asr, llm, tokenizer_fn)
#     results.append(d)

In [11]:
accumulated_dict = {}

for dictionary in results:
    for key, value in dictionary.items():
        if key in accumulated_dict:
            accumulated_dict[key] += value
        else:
            accumulated_dict[key] = value

In [94]:
keys, _ = zip(*accumulated_dict.items())
ops, poses = zip(*keys)

df = pd.DataFrame(0, columns=np.unique(ops), index=np.unique(poses))
for (op,pos), val in accumulated_dict.items():
    percnt = pos
    df.loc[pos, op] += val

df

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,116,36,2891,408
ADP,162,50,5143,489
ADV,61,26,2676,280
AUX,69,20,3148,351
CCONJ,46,4,1990,201
DET,131,34,4304,525
INTJ,1,0,175,52
MASKED,0,0,0,830
NOUN,522,89,7340,1595
NUM,17,3,230,91


In [95]:
df = df.drop(["MASKED","X","SYM","PUNCT"])#.assign(percentage=lambda df_: df_.sum(1))

df

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,116,36,2891,408
ADP,162,50,5143,489
ADV,61,26,2676,280
AUX,69,20,3148,351
CCONJ,46,4,1990,201
DET,131,34,4304,525
INTJ,1,0,175,52
NOUN,522,89,7340,1595
NUM,17,3,230,91
PART,30,9,1326,108


In [97]:
df_1 = (df.div(df.sum(1), axis=0)*100).round(2)
df_1

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
ADJ,3.36,1.04,83.77,11.82
ADP,2.77,0.86,88.00,8.37
ADV,2.00,0.85,87.94,9.20
AUX,1.92,0.56,87.74,9.78
CCONJ,2.05,0.18,88.80,8.97
DET,2.62,0.68,86.18,10.51
INTJ,0.44,0.00,76.75,22.81
NOUN,5.47,0.93,76.89,16.71
NUM,4.99,0.88,67.45,26.69
PART,2.04,0.61,90.02,7.33


In [87]:
# pctn = df.sum(0).sum()
# df_1 = df_1.assign(Percentage=(df.sum(1)/pctn).round(2)*100)

In [99]:
df_1.index = df_1.index.map(lambda idx: spacy.explain(idx))

df_1

,Improved,IntroducedError,LeftCorrect,LeftIncorrect
adjective,3.36,1.04,83.77,11.82
adposition,2.77,0.86,88.00,8.37
adverb,2.00,0.85,87.94,9.20
auxiliary,1.92,0.56,87.74,9.78
coordinating conjunction,2.05,0.18,88.80,8.97
determiner,2.62,0.68,86.18,10.51
interjection,0.44,0.00,76.75,22.81
noun,5.47,0.93,76.89,16.71
numeral,4.99,0.88,67.45,26.69
particle,2.04,0.61,90.02,7.33


In [100]:
print(df_1.to_latex(float_format="{:.2f}".format))

\begin{tabular}{lrrrr}
\toprule
 & Improved & IntroducedError & LeftCorrect & LeftIncorrect \\
\midrule
adjective & 3.36 & 1.04 & 83.77 & 11.82 \\
adposition & 2.77 & 0.86 & 88.00 & 8.37 \\
adverb & 2.00 & 0.85 & 87.94 & 9.20 \\
auxiliary & 1.92 & 0.56 & 87.74 & 9.78 \\
coordinating conjunction & 2.05 & 0.18 & 88.80 & 8.97 \\
determiner & 2.62 & 0.68 & 86.18 & 10.51 \\
interjection & 0.44 & 0.00 & 76.75 & 22.81 \\
noun & 5.47 & 0.93 & 76.89 & 16.71 \\
numeral & 4.99 & 0.88 & 67.45 & 26.69 \\
particle & 2.04 & 0.61 & 90.02 & 7.33 \\
pronoun & 1.56 & 0.51 & 89.90 & 8.04 \\
proper noun & 6.50 & 0.54 & 48.84 & 44.12 \\
subordinating conjunction & 1.43 & 0.75 & 90.16 & 7.67 \\
verb & 3.98 & 0.96 & 81.24 & 13.82 \\
\bottomrule
\end{tabular}



In [108]:
pctn = df.sum(1).sum()
sum_of_modifications = df.sum(1)
pctn, sum_of_modifications

(52111,
 ADJ      3451
 ADP      5844
 ADV      3043
 AUX      3588
 CCONJ    2241
 DET      4994
 INTJ      228
 NOUN     9546
 NUM       341
 PART     1473
 PRON     7317
 PROPN    1292
 SCONJ    1473
 VERB     7280
 dtype: int64)

In [121]:
df_2 = (df.div(df.sum(0)).assign(Percentage=sum_of_modifications/pctn)*100).round(2)
df_2

,Improved,IntroducedError,LeftCorrect,LeftIncorrect,Percentage
ADJ,6.97,9.09,6.62,6.40,6.62
ADP,9.74,12.63,11.78,7.67,11.21
ADV,3.67,6.57,6.13,4.39,5.84
AUX,4.15,5.05,7.21,5.50,6.89
CCONJ,2.76,1.01,4.56,3.15,4.30
DET,7.87,8.59,9.85,8.23,9.58
INTJ,0.06,0.00,0.40,0.82,0.44
NOUN,31.37,22.47,16.81,25.01,18.32
NUM,1.02,0.76,0.53,1.43,0.65
PART,1.80,2.27,3.04,1.69,2.83


In [124]:
df_2.index = df_2.index.map(lambda idx: spacy.explain(idx))
print(df_2.to_latex(float_format='{:0.2f}'.format))

\begin{tabular}{lrrrrr}
\toprule
 & Improved & IntroducedError & LeftCorrect & LeftIncorrect & Percentage \\
\midrule
adjective & 6.97 & 9.09 & 6.62 & 6.40 & 6.62 \\
adposition & 9.74 & 12.63 & 11.78 & 7.67 & 11.21 \\
adverb & 3.67 & 6.57 & 6.13 & 4.39 & 5.84 \\
auxiliary & 4.15 & 5.05 & 7.21 & 5.50 & 6.89 \\
coordinating conjunction & 2.76 & 1.01 & 4.56 & 3.15 & 4.30 \\
determiner & 7.87 & 8.59 & 9.85 & 8.23 & 9.58 \\
interjection & 0.06 & 0.00 & 0.40 & 0.82 & 0.44 \\
noun & 31.37 & 22.47 & 16.81 & 25.01 & 18.32 \\
numeral & 1.02 & 0.76 & 0.53 & 1.43 & 0.65 \\
particle & 1.80 & 2.27 & 3.04 & 1.69 & 2.83 \\
pronoun & 6.85 & 9.34 & 15.06 & 9.22 & 14.04 \\
proper noun & 5.05 & 1.77 & 1.44 & 8.94 & 2.48 \\
subordinating conjunction & 1.26 & 2.78 & 3.04 & 1.77 & 2.83 \\
verb & 17.43 & 17.68 & 13.54 & 15.78 & 13.97 \\
\bottomrule
\end{tabular}

